# Chapter 4: What Makes a Good Recommendation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github//modern-recommender-systems/blob/main/notebooks/chapter-04/evaluation.ipynb)

This notebook accompanies Chapter 4 of *Modern Recommender Systems*. It walks through the complete offline evaluation workflow:

1. Preparing data with temporal splitting
2. Training an ALS model (from Chapter 3)
3. Defining the relevance set
4. Calculating accuracy metrics (precision, recall, hit rate)
5. Calculating rank-aware metrics (NDCG, MAP, MRR)
6. Establishing baselines (random, popularity)
7. Beyond-accuracy metrics (coverage, diversity)
8. Comparing models

We will use the MovieLens 25M dataset.

## Setup and Installation

In [ ]:
!pip install implicit pandas numpy scipy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse
from implicit.als import AlternatingLeastSquares
from sklearn.metrics import ndcg_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Load the MovieLens Dataset

We use a subset of the MovieLens 25M dataset, filtered to active users and recent interactions to keep experiment turnaround times fast.

In [ ]:
# Download MovieLens 25M if not already present
import os
import zipfile
import urllib.request

if not os.path.exists('ml-25m'):
    print('Downloading MovieLens 25M...')
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-25m.zip',
        'ml-25m.zip'
    )
    with zipfile.ZipFile('ml-25m.zip', 'r') as z:
        z.extractall('.')
    print('Done.')
else:
    print('MovieLens 25M already downloaded.')

In [ ]:
ratings = pd.read_csv('ml-25m/ratings.csv')
movies = pd.read_csv('ml-25m/movies.csv')

# Convert timestamp to datetime for readability
ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')

print(f"Total ratings: {len(ratings):,}")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Movies: {ratings['movieId'].nunique():,}")
print(f"Date range: {ratings['date'].min()} to {ratings['date'].max()}")

## 4.3 Preparing Data for Evaluation

### 4.3.1 Temporal Splitting

We split by time: older interactions for training, recent interactions for testing. This simulates real deployment where the model learns from the past and must predict the future.

In [ ]:
# Listing 4.1 - Splitting data into train and test

ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(ratings_sorted) * 0.9)

train_val_ratings = ratings_sorted.iloc[:split_idx].copy()
test_ratings = ratings_sorted.iloc[split_idx:].copy()

In [ ]:
# Listing 4.2 - Overview of the split

print(f"Total ratings: {len(ratings_sorted):,}")
print(f"\nTemporal Split:")
print(f"  Train/Val set: {len(train_val_ratings):,} ratings ({len(train_val_ratings)/len(ratings_sorted)*100:.1f}%)")
print(f"  Test set:      {len(test_ratings):,} ratings ({len(test_ratings)/len(ratings_sorted)*100:.1f}%)")
print(f"\nDate ranges:")
print(f"  Train/Val: {train_val_ratings['date'].min()} to {train_val_ratings['date'].max()}")
print(f"  Test:      {test_ratings['date'].min()} to {test_ratings['date'].max()}")
print(f"\nUsers and movies:")
print(f"  Train/Val - Users: {train_val_ratings['userId'].nunique():,}, Movies: {train_val_ratings['movieId'].nunique():,}")
print(f"  Test      - Users: {test_ratings['userId'].nunique():,}, Movies: {test_ratings['movieId'].nunique():,}")

### 4.3.2 Start with a Sample

Start small to keep turnaround times fast. We sample a subset of users and scale up once the pipeline works.

In [ ]:
# Listing 4.3 - Sampling data

SAMPLE_FRACTION = 0.1  # Adjust: 0.1 (fast), 0.3 (moderate), 1.0 (full)

sampled_users = train_val_ratings['userId'].drop_duplicates().sample(
    frac=SAMPLE_FRACTION, random_state=42
).values

train_sampled = train_val_ratings[train_val_ratings['userId'].isin(sampled_users)].copy()
test_sampled = test_ratings[test_ratings['userId'].isin(sampled_users)].copy()

print(f"Sampled {len(sampled_users):,} users ({SAMPLE_FRACTION*100:.0f}%)")
print(f"Train ratings: {len(train_sampled):,}")
print(f"Test ratings:  {len(test_sampled):,}")

### Build User and Item Mappings

We need contiguous integer IDs for the sparse matrix. This mapping lets us go back and forth between original IDs and matrix indices.

In [ ]:
# Create mappings from original IDs to contiguous indices
all_user_ids = sorted(set(train_sampled['userId'].unique()) | set(test_sampled['userId'].unique()))
all_movie_ids = sorted(set(train_sampled['movieId'].unique()) | set(test_sampled['movieId'].unique()))

user_to_idx = {uid: idx for idx, uid in enumerate(all_user_ids)}
idx_to_user = {idx: uid for uid, idx in user_to_idx.items()}

movie_to_idx = {mid: idx for idx, mid in enumerate(all_movie_ids)}
idx_to_movie = {idx: mid for mid, idx in movie_to_idx.items()}

n_users = len(all_user_ids)
n_movies = len(all_movie_ids)

print(f"Matrix dimensions: {n_users:,} users x {n_movies:,} movies")

In [ ]:
# Build sparse user-item matrix from training data
# For implicit feedback, we use binary interactions (rating >= 4.0)
train_implicit = train_sampled[train_sampled['rating'] >= 4.0].copy()

row_indices = train_implicit['userId'].map(user_to_idx).values
col_indices = train_implicit['movieId'].map(movie_to_idx).values
values = np.ones(len(train_implicit), dtype=np.float32)

user_item_matrix = sparse.csr_matrix(
    (values, (row_indices, col_indices)),
    shape=(n_users, n_movies)
)

print(f"User-item matrix: {user_item_matrix.shape}")
print(f"Non-zero entries: {user_item_matrix.nnz:,}")
print(f"Sparsity: {1 - user_item_matrix.nnz / (n_users * n_movies):.6f}")

## Train the ALS Model

We use the same ALS model from Chapter 3. The focus of this chapter is evaluation, not model building — so we train quickly and move on to measuring how well it works.

In [ ]:
als_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    iterations=15,
    random_state=42
)

als_model.fit(user_item_matrix)
print("ALS model trained.")

## 4.4 Your First Evaluation

### 4.4.1 Generating Recommendations

In [ ]:
# Listing 4.4 - Generating recommendations from the ALS model

def generate_recommendations_als(model, user_item_csr, user_indices, k=10):
    """
    Generate recommendations for a list of users.
    
    Args:
        model: Trained ALS model
        user_item_csr: Sparse user-item interaction matrix
        user_indices: List of user indices (matrix row indices)
        k: Number of recommendations per user
    
    Returns:
        DataFrame with columns [user_idx, item_idx, rank, score]
    """
    all_recommendations = []
    
    for user_idx in user_indices:
        item_ids, scores = model.recommend(
            user_idx,
            user_item_csr[user_idx],
            N=k,
            filter_already_liked_items=True
        )
        
        for rank, (item_idx, score) in enumerate(zip(item_ids, scores), 1):
            all_recommendations.append({
                'user_idx': user_idx,
                'item_idx': item_idx,
                'rank': rank,
                'score': float(score)
            })
    
    return pd.DataFrame(all_recommendations)

In [ ]:
# Find users who appear in both train and test sets
test_user_original = test_sampled['userId'].unique()
test_user_indices = [user_to_idx[uid] for uid in test_user_original if uid in user_to_idx]

K = 10
als_recs = generate_recommendations_als(als_model, user_item_matrix, test_user_indices, k=K)

print(f"Generated {len(als_recs):,} recommendations for {len(test_user_indices):,} users")
print(f"\nExample recommendations for first user:")
first_user = test_user_indices[0]
example = als_recs[als_recs['user_idx'] == first_user].copy()
example['movie_title'] = example['item_idx'].map(
    lambda idx: movies[movies['movieId'] == idx_to_movie.get(idx, -1)]['title'].values[0]
    if idx_to_movie.get(idx, -1) in movies['movieId'].values else 'Unknown'
)
print(example[['rank', 'movie_title', 'score']].to_string(index=False))

### 4.4.2 Defining the Relevance Set

In section 4.1.1 we said there's no ground truth in recommender systems — and there isn't. What we can build is a **relevance set**: the items each user rated highly during the test period. This is our best available proxy for "the user liked this."

In [ ]:
# Listing 4.5 - Building the relevance set from test data

def build_relevance_set(test_df, user_to_idx, movie_to_idx, rating_threshold=4.0):
    """
    Extract what users actually liked in the test period.
    
    Args:
        test_df: Test ratings DataFrame
        user_to_idx: Mapping from userId to matrix index
        movie_to_idx: Mapping from movieId to matrix index
        rating_threshold: Minimum rating to consider "relevant"
    
    Returns:
        Dict mapping user_idx to set of relevant item_idx
    """
    relevant = test_df[test_df['rating'] >= rating_threshold].copy()
    relevant = relevant[relevant['movieId'].isin(movie_to_idx)]
    
    relevance_dict = defaultdict(set)
    for _, row in relevant.iterrows():
        user_idx = user_to_idx.get(row['userId'])
        item_idx = movie_to_idx.get(row['movieId'])
        if user_idx is not None and item_idx is not None:
            relevance_dict[user_idx].add(item_idx)
    
    return dict(relevance_dict)

relevance_set = build_relevance_set(test_sampled, user_to_idx, movie_to_idx, rating_threshold=4.0)

# Only evaluate users who have at least one relevant item
eval_users = [u for u in test_user_indices if u in relevance_set and len(relevance_set[u]) > 0]

print(f"Relevance set: {sum(len(v) for v in relevance_set.values()):,} relevant items")
print(f"Users with relevant items: {len(eval_users):,}")
print(f"Avg relevant items per user: {np.mean([len(relevance_set[u]) for u in eval_users]):.1f}")

## 4.4.3 Calculating Your First Metric

Let's calculate **precision@10**: what fraction of our top 10 recommendations were relevant?

In [ ]:
# Listing 4.6 - Calculating precision@k

def precision_at_k(recommended_items, relevant_items, k):
    """Precision@k for a single user."""
    top_k = recommended_items[:k]
    hits = len(set(top_k) & relevant_items)
    return hits / k

def recall_at_k(recommended_items, relevant_items, k):
    """Recall@k for a single user."""
    if len(relevant_items) == 0:
        return 0.0
    top_k = recommended_items[:k]
    hits = len(set(top_k) & relevant_items)
    return hits / len(relevant_items)

def hit_rate_at_k(recommended_items, relevant_items, k):
    """Hit rate@k for a single user: 1 if any relevant item in top-k, else 0."""
    top_k = set(recommended_items[:k])
    return 1.0 if len(top_k & relevant_items) > 0 else 0.0

In [ ]:
def get_user_recs(recs_df, user_idx):
    """Get ordered list of recommended item indices for a user."""
    user_recs = recs_df[recs_df['user_idx'] == user_idx].sort_values('rank')
    return user_recs['item_idx'].tolist()

def evaluate_accuracy_metrics(recs_df, relevance_set, eval_users, k=10):
    """
    Calculate accuracy metrics for all users.
    
    Returns:
        DataFrame with per-user precision, recall, and hit rate
    """
    results = []
    for user_idx in eval_users:
        rec_items = get_user_recs(recs_df, user_idx)
        rel_items = relevance_set.get(user_idx, set())
        
        results.append({
            'user_idx': user_idx,
            'precision': precision_at_k(rec_items, rel_items, k),
            'recall': recall_at_k(rec_items, rel_items, k),
            'hit_rate': hit_rate_at_k(rec_items, rel_items, k),
            'num_relevant': len(rel_items)
        })
    
    return pd.DataFrame(results)

als_accuracy = evaluate_accuracy_metrics(als_recs, relevance_set, eval_users, k=K)

print(f"ALS Model - Accuracy Metrics @{K}")
print(f"  Mean Precision:  {als_accuracy['precision'].mean():.4f}")
print(f"  Mean Recall:     {als_accuracy['recall'].mean():.4f}")
print(f"  Mean Hit Rate:   {als_accuracy['hit_rate'].mean():.4f}")
print(f"  Median Precision: {als_accuracy['precision'].median():.4f}")
print(f"  Users evaluated: {len(als_accuracy):,}")

## 4.4.4 Adding Baselines

### Random Baseline

In [ ]:
def generate_recommendations_random(all_item_indices, user_indices, k=10, seed=42):
    """
    Generate random recommendations: pick k items uniformly from the catalog.
    """
    rng = np.random.RandomState(seed)
    recommendations = []
    
    for user_idx in user_indices:
        random_items = rng.choice(all_item_indices, size=k, replace=False)
        for rank, item_idx in enumerate(random_items, 1):
            recommendations.append({
                'user_idx': user_idx,
                'item_idx': item_idx,
                'rank': rank,
                'score': 0.0
            })
    
    return pd.DataFrame(recommendations)

all_item_indices = list(range(n_movies))
random_recs = generate_recommendations_random(all_item_indices, eval_users, k=K)
random_accuracy = evaluate_accuracy_metrics(random_recs, relevance_set, eval_users, k=K)

print(f"Random Baseline - Accuracy Metrics @{K}")
print(f"  Mean Precision: {random_accuracy['precision'].mean():.4f}")
print(f"  Mean Recall:    {random_accuracy['recall'].mean():.4f}")
print(f"  Mean Hit Rate:  {random_accuracy['hit_rate'].mean():.4f}")

### Popularity Baseline

In [ ]:
# Listing 4.7 - Popularity baseline

def generate_recommendations_popularity(train_df, movie_to_idx, user_indices, k=10):
    """
    Recommend the k most popular items to everyone.
    Popularity = number of interactions in training data.
    """
    # Count interactions per movie, map to indices
    item_counts = train_df['movieId'].value_counts()
    top_items = []
    for movie_id, count in item_counts.items():
        if movie_id in movie_to_idx:
            top_items.append((movie_to_idx[movie_id], count))
        if len(top_items) >= k:
            break
    
    recommendations = []
    for user_idx in user_indices:
        for rank, (item_idx, count) in enumerate(top_items, 1):
            recommendations.append({
                'user_idx': user_idx,
                'item_idx': item_idx,
                'rank': rank,
                'score': float(count)
            })
    
    return pd.DataFrame(recommendations)

pop_recs = generate_recommendations_popularity(train_sampled, movie_to_idx, eval_users, k=K)
pop_accuracy = evaluate_accuracy_metrics(pop_recs, relevance_set, eval_users, k=K)

print(f"Popularity Baseline - Accuracy Metrics @{K}")
print(f"  Mean Precision: {pop_accuracy['precision'].mean():.4f}")
print(f"  Mean Recall:    {pop_accuracy['recall'].mean():.4f}")
print(f"  Mean Hit Rate:  {pop_accuracy['hit_rate'].mean():.4f}")

### Compare Accuracy Metrics

In [ ]:
def compare_models_accuracy(metrics_dict, k):
    """Print a comparison table of accuracy metrics across models."""
    print(f"\n{'='*65}")
    print(f"MODEL COMPARISON: Accuracy Metrics @{k}")
    print(f"{'='*65}")
    print(f"{'Model':<20} {'Precision':<12} {'Recall':<12} {'Hit Rate':<12}")
    print(f"{'-'*65}")
    
    for name, metrics_df in metrics_dict.items():
        print(f"{name:<20} "
              f"{metrics_df['precision'].mean():<12.4f} "
              f"{metrics_df['recall'].mean():<12.4f} "
              f"{metrics_df['hit_rate'].mean():<12.4f}")
    print(f"{'='*65}")
    
    # Show improvement of ALS over popularity
    if 'Popularity' in metrics_dict and 'ALS' in metrics_dict:
        pop_prec = metrics_dict['Popularity']['precision'].mean()
        als_prec = metrics_dict['ALS']['precision'].mean()
        if pop_prec > 0:
            improvement = (als_prec - pop_prec) / pop_prec * 100
            print(f"\nALS improvement over Popularity (precision): {improvement:+.1f}%")

compare_models_accuracy({
    'Random': random_accuracy,
    'Popularity': pop_accuracy,
    'ALS': als_accuracy
}, K)

## 4.6.2 Averages and Distributions

Mean precision is just one number — it can hide very different realities. Let's look at the distribution of precision across users.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, metrics_df) in zip(axes, [
    ('Random', random_accuracy),
    ('Popularity', pop_accuracy),
    ('ALS', als_accuracy)
]):
    ax.hist(metrics_df['precision'], bins=20, edgecolor='black', alpha=0.7)
    ax.axvline(metrics_df['precision'].mean(), color='red', linestyle='--',
               label=f"Mean: {metrics_df['precision'].mean():.4f}")
    ax.axvline(metrics_df['precision'].median(), color='blue', linestyle=':',
               label=f"Median: {metrics_df['precision'].median():.4f}")
    ax.set_title(f'{name}')
    ax.set_xlabel('Precision@10')
    ax.set_ylabel('Number of Users')
    ax.legend(fontsize=8)

plt.suptitle('Distribution of Precision@10 Across Users', fontsize=14)
plt.tight_layout()
plt.show()

## 4.6.3 Rank-Aware Metrics

Precision tells us how many relevant items are in the list, but not where they are. Rank-aware metrics reward systems that place relevant items near the top.

In [ ]:
def ndcg_at_k(recommended_items, relevant_items, k):
    """NDCG@k for a single user."""
    top_k = recommended_items[:k]
    relevance = [1.0 if item in relevant_items else 0.0 for item in top_k]
    
    if sum(relevance) == 0:
        return 0.0
    
    # DCG
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevance))
    
    # IDCG: ideal ranking has all relevant items at the top
    ideal_relevance = sorted(relevance, reverse=True)
    n_relevant = min(len(relevant_items), k)
    ideal_relevance = [1.0] * n_relevant + [0.0] * (k - n_relevant)
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevance))
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg


def mrr_at_k(recommended_items, relevant_items, k):
    """Reciprocal Rank@k for a single user."""
    top_k = recommended_items[:k]
    for i, item in enumerate(top_k):
        if item in relevant_items:
            return 1.0 / (i + 1)
    return 0.0


def average_precision_at_k(recommended_items, relevant_items, k):
    """Average Precision@k for a single user."""
    top_k = recommended_items[:k]
    hits = 0
    sum_precision = 0.0
    
    for i, item in enumerate(top_k):
        if item in relevant_items:
            hits += 1
            sum_precision += hits / (i + 1)
    
    if hits == 0:
        return 0.0
    
    return sum_precision / min(len(relevant_items), k)

In [ ]:
def evaluate_ranking_metrics(recs_df, relevance_set, eval_users, k=10):
    """
    Calculate ranking metrics for all users.
    
    Returns:
        DataFrame with per-user NDCG, MRR, and MAP
    """
    results = []
    for user_idx in eval_users:
        rec_items = get_user_recs(recs_df, user_idx)
        rel_items = relevance_set.get(user_idx, set())
        
        results.append({
            'user_idx': user_idx,
            'ndcg': ndcg_at_k(rec_items, rel_items, k),
            'mrr': mrr_at_k(rec_items, rel_items, k),
            'map': average_precision_at_k(rec_items, rel_items, k)
        })
    
    return pd.DataFrame(results)

# Calculate ranking metrics for all models
als_ranking = evaluate_ranking_metrics(als_recs, relevance_set, eval_users, k=K)
pop_ranking = evaluate_ranking_metrics(pop_recs, relevance_set, eval_users, k=K)
random_ranking = evaluate_ranking_metrics(random_recs, relevance_set, eval_users, k=K)

print(f"\n{'='*65}")
print(f"MODEL COMPARISON: Ranking Metrics @{K}")
print(f"{'='*65}")
print(f"{'Model':<20} {'NDCG':<12} {'MRR':<12} {'MAP':<12}")
print(f"{'-'*65}")

for name, metrics_df in [('Random', random_ranking), ('Popularity', pop_ranking), ('ALS', als_ranking)]:
    print(f"{name:<20} "
          f"{metrics_df['ndcg'].mean():<12.4f} "
          f"{metrics_df['mrr'].mean():<12.4f} "
          f"{metrics_df['map'].mean():<12.4f}")

print(f"{'='*65}")

## Metrics Across Different K Values

How do metrics change as we recommend more items? This shows the trade-offs between precision (which tends to decrease with larger K) and recall (which tends to increase).

In [ ]:
# Generate recommendations at larger K for this analysis
K_MAX = 20
als_recs_large = generate_recommendations_als(als_model, user_item_matrix, eval_users, k=K_MAX)
pop_recs_large = generate_recommendations_popularity(train_sampled, movie_to_idx, eval_users, k=K_MAX)

k_values = [5, 10, 15, 20]
metrics_by_k = {'ALS': {}, 'Popularity': {}}

for k in k_values:
    als_acc_k = evaluate_accuracy_metrics(als_recs_large, relevance_set, eval_users, k=k)
    als_rank_k = evaluate_ranking_metrics(als_recs_large, relevance_set, eval_users, k=k)
    metrics_by_k['ALS'][k] = {
        'precision': als_acc_k['precision'].mean(),
        'recall': als_acc_k['recall'].mean(),
        'ndcg': als_rank_k['ndcg'].mean(),
        'mrr': als_rank_k['mrr'].mean(),
        'map': als_rank_k['map'].mean()
    }
    
    pop_acc_k = evaluate_accuracy_metrics(pop_recs_large, relevance_set, eval_users, k=k)
    pop_rank_k = evaluate_ranking_metrics(pop_recs_large, relevance_set, eval_users, k=k)
    metrics_by_k['Popularity'][k] = {
        'precision': pop_acc_k['precision'].mean(),
        'recall': pop_acc_k['recall'].mean(),
        'ndcg': pop_rank_k['ndcg'].mean(),
        'mrr': pop_rank_k['mrr'].mean(),
        'map': pop_rank_k['map'].mean()
    }

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
metric_names = ['precision', 'recall', 'ndcg', 'mrr', 'map']
metric_labels = ['Precision', 'Recall', 'NDCG', 'MRR', 'MAP']

for ax, metric, label in zip(axes, metric_names, metric_labels):
    for model_name, color in [('ALS', 'steelblue'), ('Popularity', 'coral')]:
        values = [metrics_by_k[model_name][k][metric] for k in k_values]
        ax.plot(k_values, values, 'o-', label=model_name, color=color)
    
    ax.set_xlabel('K (Top-K Recommendations)')
    ax.set_ylabel(f'{label} Score')
    ax.set_title(f'{label} vs K')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Metrics Across Different K Values', fontsize=14)
plt.tight_layout()
plt.show()

## 4.6.4 Beyond Accuracy

Accuracy and ranking metrics tell you whether the system recommends items users engage with. But a system can score well on precision and NDCG while still failing its users.

### User Coverage: Hit Rate by User Segment

In [ ]:
# Segment users by their activity level in the training data
user_activity = train_sampled.groupby('userId').size().reset_index(name='n_ratings')
user_activity['user_idx'] = user_activity['userId'].map(user_to_idx)

# Define segments based on quartiles
quartiles = user_activity['n_ratings'].quantile([0.25, 0.5, 0.75])

def assign_segment(n):
    if n <= quartiles[0.25]:
        return 'Cold-Start Users'
    elif n <= quartiles[0.50]:
        return 'Occasional Users'
    elif n <= quartiles[0.75]:
        return 'Regular Users'
    else:
        return 'Power Users'

user_activity['segment'] = user_activity['n_ratings'].apply(assign_segment)

# Calculate hit rate per segment for ALS
als_with_segment = als_accuracy.merge(
    user_activity[['user_idx', 'segment', 'n_ratings']],
    on='user_idx',
    how='left'
)

segment_order = ['Power Users', 'Regular Users', 'Occasional Users', 'Cold-Start Users']
segment_stats = als_with_segment.groupby('segment').agg(
    mean_hit_rate=('hit_rate', 'mean'),
    mean_precision=('precision', 'mean'),
    n_users=('user_idx', 'count')
).reindex(segment_order)

print("Hit Rate by User Segment (ALS Model):")
print(segment_stats.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

colors = ['#4CAF50', '#2196F3', '#FF9800', '#f44336']
bars = ax.barh(range(len(segment_order)), segment_stats['mean_hit_rate'], color=colors)
ax.set_yticks(range(len(segment_order)))
ax.set_yticklabels(segment_order)
ax.set_xlabel('Hit Rate@10')
ax.set_title('Hit Rate by User Segment\n(% users with ≥1 relevant item in top 10)')

for i, (v, n) in enumerate(zip(segment_stats['mean_hit_rate'], segment_stats['n_users'])):
    ax.text(v + 0.005, i, f'{v:.3f} ({v*100:.1f}%)', va='center', fontsize=10)

ax.set_xlim(0, max(segment_stats['mean_hit_rate']) * 1.3)
plt.tight_layout()
plt.show()

### Item Coverage: What Fraction of the Catalog Gets Recommended?

In [ ]:
def calculate_item_coverage(recs_df, total_items, train_df=None, movie_to_idx=None):
    """
    Analyze item coverage of recommendations.
    
    Returns:
        Dict with coverage statistics
    """
    recommended_items = set(recs_df['item_idx'].unique())
    coverage = len(recommended_items) / total_items
    
    result = {
        'n_recommended': len(recommended_items),
        'n_total': total_items,
        'coverage': coverage,
        'n_never_recommended': total_items - len(recommended_items)
    }
    
    # Analyze by popularity tier if training data provided
    if train_df is not None and movie_to_idx is not None:
        item_popularity = train_df['movieId'].value_counts()
        item_pop_idx = {movie_to_idx[mid]: count 
                        for mid, count in item_popularity.items() 
                        if mid in movie_to_idx}
        
        pop_values = sorted(item_pop_idx.values(), reverse=True)
        if pop_values:
            # Build 10 percentile thresholds (90th down to 10th)
            percentile_thresholds = [
                np.percentile(pop_values, 100 - p) for p in range(10, 100, 10)
            ]
            
            tier_labels = [
                f'Top {p*10}-{p*10+10}%' if p > 0 else 'Top 0-10%'
                for p in range(10)
            ]
            tier_labels[0] = 'Top 0-10%'
            tiers = {label: 0 for label in tier_labels}
            
            for item_idx in recommended_items:
                pop = item_pop_idx.get(item_idx, 0)
                # Find which decile this item falls into
                assigned = False
                for i, threshold in enumerate(percentile_thresholds):
                    if pop >= threshold:
                        tiers[tier_labels[i]] += 1
                        assigned = True
                        break
                if not assigned:
                    tiers[tier_labels[-1]] += 1
            
            result['tiers'] = tiers
    
    return result

als_coverage = calculate_item_coverage(als_recs, n_movies, train_sampled, movie_to_idx)

print(f"Item Coverage Analysis (ALS Model):")
print(f"  Recommended items: {als_coverage['n_recommended']:,} / {als_coverage['n_total']:,}")
print(f"  Catalog coverage:  {als_coverage['coverage']*100:.1f}%")
print(f"  Never recommended: {als_coverage['n_never_recommended']:,}")
if 'tiers' in als_coverage:
    print(f"\n  By popularity tier (10th percentile buckets):")
    for tier, count in als_coverage['tiers'].items():
        pct = count / als_coverage['n_recommended'] * 100 if als_coverage['n_recommended'] > 0 else 0
        print(f"    {tier}: {count} ({pct:.1f}%)")


In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(7, 5))

# Pie chart: recommended vs not
ax1.pie(
    [als_coverage['n_recommended'], als_coverage['n_never_recommended']],
    labels=['Recommended', 'Not Recommended'],
    autopct='%1.1f%%',
    colors=['#2196F3', '#E0E0E0'],
    startangle=90
)
ax1.set_title(f'Item Coverage in Recommendations\n'
              f'{als_coverage["n_recommended"]:,} items ({als_coverage["coverage"]*100:.1f}%)')

plt.suptitle('Recommendation System Coverage Analysis', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(12, 5))

# Bar chart: by popularity tier
if 'tiers' in als_coverage:
    tiers = als_coverage['tiers']
    tier_names = list(tiers.keys())
    tier_values = list(tiers.values())
    tier_colors = plt.cm.RdYlGn(np.linspace(0.9, 0.1, len(tier_names)))
    
    bars = ax1.bar(range(len(tier_names)), tier_values, color=tier_colors)
    ax1.set_xticks(range(len(tier_names)))
    ax1.set_xticklabels(tier_names, rotation=30, ha='right')
    ax1.set_ylabel('Number of Unique Items')
    ax1.set_title('Recommended Items by Popularity Tier (10th Percentile Buckets)')
    
    for bar, val in zip(bars, tier_values):
        pct = val / als_coverage['n_recommended'] * 100 if als_coverage['n_recommended'] > 0 else 0
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=8)

plt.suptitle('Recommendation System Coverage Analysis', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Build item popularity lookup (train interactions per item, by matrix index)
item_popularity = train_sampled['movieId'].value_counts()
item_pop_by_idx = {movie_to_idx[mid]: cnt for mid, cnt in item_popularity.items() if mid in movie_to_idx}

all_pop   = np.array([item_pop_by_idx.get(i, 0) for i in range(n_movies)], dtype=float)
rec_items = set(als_recs['item_idx'].unique())
rec_pop   = np.array([item_pop_by_idx.get(i, 0) for i in rec_items if i in item_pop_by_idx], dtype=float)

# Concentration curve: sort items by popularity desc, accumulate share of total interactions
sorted_pop = np.sort(all_pop)[::-1]
cum_recs   = np.cumsum(sorted_pop) / sorted_pop.sum()
item_pct   = np.linspace(0, 100, len(sorted_pop))

fig, axes = plt.subplots(3, 1, figsize=(10, 15))

# --- Panel 1: Popularity distribution (log scale) ---
ax = axes[0]
bins = np.logspace(np.log10(max(all_pop[all_pop > 0].min(), 1)),
                   np.log10(all_pop.max()), 40)
ax.hist(all_pop[all_pop > 0], bins=bins, alpha=0.5, color='#90CAF9', label='All catalog items')
ax.hist(rec_pop[rec_pop > 0],  bins=bins, alpha=0.7, color='#1565C0', label='Recommended items')
ax.set_xscale('log')
ax.set_xlabel('Number of Interactions (log scale)', fontsize=12)
ax.set_ylabel('Number of Items', fontsize=12)
ax.set_title('Popularity Distribution: All Catalog vs Recommended Items', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# --- Panel 2: Concentration curve ---
ax = axes[1]
ax.plot(item_pct, cum_recs * 100, color='#1565C0', linewidth=2, label='ALS')
ax.plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Perfect equality')
ax.fill_between(item_pct, cum_recs * 100, item_pct, alpha=0.1, color='#1565C0')
ax.set_xlabel('Top X% of Items (by popularity)', fontsize=12)
ax.set_ylabel('Cumulative % of All Interactions', fontsize=12)
ax.set_title('Interaction Concentration Curve', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

top10_idx = int(len(sorted_pop) * 0.10)
top10_share = sorted_pop[:top10_idx].sum() / sorted_pop.sum() * 100
ax.annotate(f'Top 10% of items\n→ {top10_share:.0f}% of interactions',
            xy=(10, top10_share), xytext=(25, top10_share - 15),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10)

# --- Panel 3: Recommended vs not recommended by popularity decile ---
ax = axes[2]
decile_edges = np.percentile(all_pop[all_pop > 0], np.arange(0, 110, 10))
rec_set = set(als_recs['item_idx'].unique())
rec_counts   = np.zeros(10)
unrec_counts = np.zeros(10)

for idx in range(n_movies):
    pop = item_pop_by_idx.get(idx, 0)
    if pop == 0:
        continue
    bucket = min(int(np.searchsorted(decile_edges[1:-1], pop)), 9)
    if idx in rec_set:
        rec_counts[bucket] += 1
    else:
        unrec_counts[bucket] += 1

x = np.arange(10)
width = 0.6
ax.bar(x, rec_counts,   width, label='Recommended',     color='#1565C0', alpha=0.8)
ax.bar(x, unrec_counts, width, bottom=rec_counts, label='Not Recommended', color='#E0E0E0', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'D{i+1}' for i in range(10)], fontsize=11)
ax.set_xlabel('Popularity Decile (D1 = least popular, D10 = most popular)', fontsize=12)
ax.set_ylabel('Number of Items', fontsize=12)
ax.set_title('Recommended vs Not Recommended by Popularity Decile', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'Coverage Analysis — {als_coverage["coverage"]*100:.1f}% of catalog recommended '
    f'({als_coverage["n_recommended"]:,} / {als_coverage["n_total"]:,} items)',
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.show()


### Intra-List Diversity

In [ ]:
def calculate_intra_list_diversity_genres(recs_df, eval_users, idx_to_movie, movies_df, k=10):
    """
    Calculate intra-list diversity based on genre overlap.
    Diversity = 1 - average Jaccard similarity between pairs of items.
    """
    # Build genre sets for each movie
    movie_genres = {}
    for _, row in movies_df.iterrows():
        if pd.notna(row['genres']):
            movie_genres[row['movieId']] = set(row['genres'].split('|'))
    
    diversities = []
    for user_idx in eval_users:
        rec_items = get_user_recs(recs_df, user_idx)[:k]
        
        # Get genres for recommended items
        rec_genres = []
        for item_idx in rec_items:
            movie_id = idx_to_movie.get(item_idx)
            if movie_id and movie_id in movie_genres:
                rec_genres.append(movie_genres[movie_id])
        
        if len(rec_genres) < 2:
            continue
        
        # Average pairwise Jaccard distance
        distances = []
        for i in range(len(rec_genres)):
            for j in range(i + 1, len(rec_genres)):
                intersection = len(rec_genres[i] & rec_genres[j])
                union = len(rec_genres[i] | rec_genres[j])
                if union > 0:
                    distances.append(1 - intersection / union)
        
        if distances:
            diversities.append(np.mean(distances))
    
    return diversities

als_diversity = calculate_intra_list_diversity_genres(als_recs, eval_users, idx_to_movie, movies, k=K)
pop_diversity = calculate_intra_list_diversity_genres(pop_recs, eval_users, idx_to_movie, movies, k=K)

print(f"Intra-List Diversity (genre-based Jaccard distance):")
print(f"  ALS:        Mean={np.mean(als_diversity):.4f}, Median={np.median(als_diversity):.4f}")
print(f"  Popularity: Mean={np.mean(pop_diversity):.4f}, Median={np.median(pop_diversity):.4f}")

In [ ]:
# Category coverage: how many unique genres appear per user's recommendations
def category_coverage(recs_df, eval_users, idx_to_movie, movies_df, k=10):
    """Count unique genres per recommendation list."""
    movie_genres = {}
    for _, row in movies_df.iterrows():
        if pd.notna(row['genres']):
            movie_genres[row['movieId']] = set(row['genres'].split('|'))
    
    all_genres = set()
    for g in movie_genres.values():
        all_genres |= g
    
    coverages = []
    for user_idx in eval_users:
        rec_items = get_user_recs(recs_df, user_idx)[:k]
        genres_in_list = set()
        for item_idx in rec_items:
            movie_id = idx_to_movie.get(item_idx)
            if movie_id and movie_id in movie_genres:
                genres_in_list |= movie_genres[movie_id]
        coverages.append(len(genres_in_list))
    
    return coverages

als_cat_cov = category_coverage(als_recs, eval_users, idx_to_movie, movies, k=K)
pop_cat_cov = category_coverage(pop_recs, eval_users, idx_to_movie, movies, k=K)

print(f"Category Coverage (unique genres per recommendation list):")
print(f"  ALS:        Mean={np.mean(als_cat_cov):.1f} genres")
print(f"  Popularity: Mean={np.mean(pop_cat_cov):.1f} genres")

## 4.9 Comparing Models: Full Dashboard

Let's bring everything together in a single comparison.

In [ ]:
# Comprehensive comparison
pop_coverage = calculate_item_coverage(pop_recs, n_movies, train_sampled, movie_to_idx)

print(f"\n{'='*75}")
print(f"COMPREHENSIVE MODEL COMPARISON @{K}")
print(f"{'='*75}")
print(f"{'Metric':<30} {'Random':<15} {'Popularity':<15} {'ALS':<15}")
print(f"{'-'*75}")

# Accuracy
print(f"{'Precision@10':<30} "
      f"{random_accuracy['precision'].mean():<15.4f} "
      f"{pop_accuracy['precision'].mean():<15.4f} "
      f"{als_accuracy['precision'].mean():<15.4f}")

print(f"{'Recall@10':<30} "
      f"{random_accuracy['recall'].mean():<15.4f} "
      f"{pop_accuracy['recall'].mean():<15.4f} "
      f"{als_accuracy['recall'].mean():<15.4f}")

print(f"{'Hit Rate@10':<30} "
      f"{random_accuracy['hit_rate'].mean():<15.4f} "
      f"{pop_accuracy['hit_rate'].mean():<15.4f} "
      f"{als_accuracy['hit_rate'].mean():<15.4f}")

# Ranking
print(f"{'-'*75}")
print(f"{'NDCG@10':<30} "
      f"{random_ranking['ndcg'].mean():<15.4f} "
      f"{pop_ranking['ndcg'].mean():<15.4f} "
      f"{als_ranking['ndcg'].mean():<15.4f}")

print(f"{'MRR@10':<30} "
      f"{random_ranking['mrr'].mean():<15.4f} "
      f"{pop_ranking['mrr'].mean():<15.4f} "
      f"{als_ranking['mrr'].mean():<15.4f}")

print(f"{'MAP@10':<30} "
      f"{random_ranking['map'].mean():<15.4f} "
      f"{pop_ranking['map'].mean():<15.4f} "
      f"{als_ranking['map'].mean():<15.4f}")

# Beyond Accuracy
print(f"{'-'*75}")
print(f"{'Catalog Coverage (%)':<30} "
      f"{'100.0':<15} "
      f"{pop_coverage['coverage']*100:<15.1f} "
      f"{als_coverage['coverage']*100:<15.1f}")

print(f"{'Unique Items Recommended':<30} "
      f"{n_movies:<15,} "
      f"{pop_coverage['n_recommended']:<15,} "
      f"{als_coverage['n_recommended']:<15,}")

print(f"{'Intra-List Diversity':<30} "
      f"{'N/A':<15} "
      f"{np.mean(pop_diversity):<15.4f} "
      f"{np.mean(als_diversity):<15.4f}")

print(f"{'Avg Genres per List':<30} "
      f"{'N/A':<15} "
      f"{np.mean(pop_cat_cov):<15.1f} "
      f"{np.mean(als_cat_cov):<15.1f}")

print(f"{'='*75}")

## Spot Checks: Sanity Testing with Known Items

Before trusting the metrics, it helps to look at actual recommendations for items you understand. Pick a well-known movie and check whether the recommendations make sense.

In [ ]:
def spot_check_item(model, user_item_matrix, movie_title, movies_df, movie_to_idx, idx_to_movie, k=10):
    """
    Find a movie by title and show its most similar items according to the model.
    This is a quick sanity check: do the similar items make sense?
    """
    # Find the movie
    matches = movies_df[movies_df['title'].str.contains(movie_title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No movie found matching '{movie_title}'")
        return
    
    movie = matches.iloc[0]
    movie_id = movie['movieId']
    
    if movie_id not in movie_to_idx:
        print(f"Movie '{movie['title']}' not in the training data")
        return
    
    item_idx = movie_to_idx[movie_id]
    
    print(f"Seed: {movie['title']} ({movie['genres']})")
    print(f"{'─'*60}")
    
    # Get similar items from the ALS model
    similar_ids, scores = model.similar_items(item_idx, N=k+1)
    
    for rank, (sim_idx, score) in enumerate(zip(similar_ids[1:], scores[1:]), 1):
        sim_movie_id = idx_to_movie.get(sim_idx)
        if sim_movie_id:
            sim_movie = movies_df[movies_df['movieId'] == sim_movie_id]
            if len(sim_movie) > 0:
                title = sim_movie.iloc[0]['title']
                genres = sim_movie.iloc[0]['genres']
                print(f"  {rank}. {title} ({genres}) - score: {score:.4f}")

# Try a few well-known movies
print("=" * 60)
print("SPOT CHECKS: Similar Items")
print("=" * 60)

for title in ['Star Wars', 'Toy Story', 'The Godfather']:
    print()
    spot_check_item(als_model, user_item_matrix, title, movies, movie_to_idx, idx_to_movie, k=5)
    print()